### Schritt 1: Dokument load, z.B. PDFs oder Texte, Splitten in Chunks

In [3]:
import importlib
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import pandas as pd
import rag_tutorial.rag_utils
importlib.reload(rag_tutorial.rag_utils)
from rag_tutorial.rag_utils import create_rag_system, build_rag_chain, create_judge_llm, make_judges
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    GEval
)
from deepeval.test_case import LLMTestCase, SingleTurnParams

In [4]:
# Create sample files for testing

# Projektstamm anhand von pyproject.toml finden, unabhängig vom Arbeitsverzeichnis des Notebooks
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Project root: {project_root}")
documents_dir = project_root / "documents"
documents_dir.mkdir(exist_ok=True)
print(f"Documents dir: {documents_dir}")

Project root: c:\Artificial Intelligence\rag_tutorial
Documents dir: c:\Artificial Intelligence\rag_tutorial\documents


In [5]:
loader = DirectoryLoader(
    str(documents_dir),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

In [6]:
from langchain_core.documents import Document

In [7]:
raw_documents: list [Document]= loader.load()
print(f"Loaded {len(raw_documents)} documents")
for doc in raw_documents:
    source = doc.metadata.get("source", "unknown")
    print(f"{source + ':':<28} {len(doc.page_content)} characters")

Loaded 10 documents
c:\Artificial Intelligence\rag_tutorial\documents\ai-intro.txt: 30133 characters
c:\Artificial Intelligence\rag_tutorial\documents\encoding_problems.txt: 593 characters
c:\Artificial Intelligence\rag_tutorial\documents\messy_report.txt: 609 characters
c:\Artificial Intelligence\rag_tutorial\documents\ml-practical-guide.txt: 12221 characters
c:\Artificial Intelligence\rag_tutorial\documents\neural-networks.txt: 11125 characters
c:\Artificial Intelligence\rag_tutorial\documents\python-classes.txt: 3218 characters
c:\Artificial Intelligence\rag_tutorial\documents\python-functions.txt: 2619 characters
c:\Artificial Intelligence\rag_tutorial\documents\python-intro.txt: 3849 characters
c:\Artificial Intelligence\rag_tutorial\documents\scraped_content.txt: 693 characters
c:\Artificial Intelligence\rag_tutorial\documents\short.txt: 59 characters


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [9]:
documents = text_splitter.split_documents(raw_documents)
print(f"Split into {len(documents)} chunks")


Split into 91 chunks


In [10]:
print(f"First document:\n{documents[0].page_content}\n \ndocument metadata:\n{documents[0].metadata}")

First document:
# Introduction to AI and ML

## I. Definitions and Conceptual Foundations

Artificial Intelligence (AI) refers to the interdisciplinary scientific field
devoted to creating machines capable of performing tasks that typically require
human intelligence. These tasks encompass perception (computer vision, speech
recognition), reasoning (logical inference, problem-solving), learning (pattern
recognition, adaptation), and decision-making under uncertainty. However, the
definition of AI remains fluid; as computers master specific capabilities—such
as arithmetic, chess, or facial recognition—these tasks often cease to be
classified as "artificial intelligence" in popular discourse and instead become
mere "computation." This phenomenon, known as the **AI effect**, suggests that
intelligence is frequently defined as whatever machines cannot yet do.
 
document metadata:
{'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.txt'}


In [11]:
""" print source and content for all documents, first 500 chars with indexing"""
for i, doc in enumerate(documents):
    print(f"{i}: {doc.metadata.get('source', 'unknown')}")
    print(doc.page_content[:500])
    print()


0: c:\Artificial Intelligence\rag_tutorial\documents\ai-intro.txt
# Introduction to AI and ML

## I. Definitions and Conceptual Foundations

Artificial Intelligence (AI) refers to the interdisciplinary scientific field
devoted to creating machines capable of performing tasks that typically require
human intelligence. These tasks encompass perception (computer vision, speech
recognition), reasoning (logical inference, problem-solving), learning (pattern
recognition, adaptation), and decision-making under uncertainty. However, the
definition of AI remains fluid;

1: c:\Artificial Intelligence\rag_tutorial\documents\ai-intro.txt
Within AI research, a fundamental distinction separates **Strong AI** (or
Artificial General Intelligence, AGI) from **Weak AI** (or Narrow AI). Strong AI
denotes hypothetical systems possessing generalized cognitive abilities
comparable to human intelligence—capable of transferring knowledge across
domains, understanding context, and exhibiting consciousness or s

### Schritt 2: RAG build and evaluation using LLM judge

In [12]:
# reference sind +.txt Dokumente in Ordner documents    
test_set = [
    {"question": "Was ist Overfitting?",
     "reference": "Overfitting tritt auf, wenn ein Modell die Trainingsdaten "
     "auswendig lernt und nicht auf neue Daten generalisiert."},
    {"question": "Was ist der Unterschied zwischen L1- und L2-Regularisierung?",
     "reference": "L1-Regularisierung (Lasso) bestraft die Absolutwerte der Gewichte und "
     "setzt manche Gewichte auf exakt null, wirkt also wie Feature-Selektion. "
     "L2-Regularisierung (Ridge) bestraft die quadrierten Gewichte und erzeugt kleine, "
     "aber von null verschiedene Werte."},
    {"question": "Warum ist Accuracy bei unausgewogenen Klassen irreführend?",
     "reference": "Bei stark unausgewogenen Klassen erreicht ein Modell, das immer die "
     "Mehrheitsklasse vorhersagt, eine hohe Accuracy (z. B. 95 % bei Betrugserkennung), "
     "erkennt aber die seltene Klasse überhaupt nicht."},
    {"question": "Was ist Backpropagation?",
     "reference": "Backpropagation wendet die Kettenregel rückwärts durch das Netz an und "
     "berechnet den Gradienten des Loss bezüglich jedes Gewichts. Damit wird bestimmt, "
     "wie stark jedes Gewicht zum Fehler beigetragen hat."},
    {"question": "Welches Problem haben einfache RNNs bei langen Sequenzen?",
     "reference": "Einfache RNNs leiden unter dem Vanishing-Gradient-Problem: Die Gradienten "
     "schrumpfen beim Backpropagation durch viele Zeitschritte exponentiell, sodass "
     "langfristige Muster nicht gelernt werden. LSTM und GRU lösen das mit Gating-Mechanismen."},
    {"question": "Wofür werden Convolutional Neural Networks (CNNs) eingesetzt?",
     "reference": "CNNs sind für gitterstrukturierte Daten wie Bilder gedacht. Kleine Filter "
     "gleiten über die Eingabe und erkennen lokale Merkmale wie Kanten und Texturen; "
     "gestapelte Schichten bilden eine Hierarchie bis hin zu komplexen Objekten."},
    {"question": "Was ist Dropout?",
     "reference": "Dropout deaktiviert in jedem Trainingsschritt zufällig einen Teil der "
     "Neuronen. Das erzwingt redundante Repräsentationen und verhindert, dass einzelne "
     "Neuronen zu stark spezialisiert werden; es ist eine Regularisierungstechnik gegen Overfitting."},
    {"question": "Was bedeutet Data Drift und Concept Drift?",
     "reference": "Data Drift bedeutet, dass sich die statistische Verteilung der "
     "eingehenden Daten über die Zeit ändert. Concept Drift bedeutet, dass sich die "
     "Beziehung zwischen Eingaben und Zielvariable ändert. Beides verschlechtert "
     "die Modellgenauigkeit."},
    {"question": "Was ist ein Standardwert (Default-Argument) in Python-Funktionen und welche Falle gibt es?",
     "reference": "Default-Werte werden bei der Funktionsdefinition ausgewertet, nicht beim "
     "Aufruf. Veränderliche Defaults wie eine leere Liste bleiben deshalb über mehrere "
     "Aufrufe hinweg bestehen, was zu unerwarteten Seiteneffekten führt."},
    {"question": "Was ist Transfer Learning?",
     "reference": "Beim Transfer Learning startet man mit einem auf einem großen, allgemeinen "
     "Datensatz vorab trainierten Modell und passt es per Fine-Tuning auf einen kleineren, "
     "aufgabenspezifischen Datensatz an. Das reduziert den Bedarf an Daten und Rechenleistung erheblich."},
]


In [13]:
LANGUAGE = "de"
_, _, vectorstore, llm, judge_llm = create_rag_system(
    k=2, docs_content = documents, collection_name="ml_docs_run", language=LANGUAGE
)

In [14]:
judge_faithfulness, judge_relevance, judge_correctness = make_judges(judge_llm)

In [15]:
def evaluate(k):
    retriever, chain = build_rag_chain(vectorstore, llm, k, language=LANGUAGE)
    rows = []
    for item in test_set:
        q, ref = item["question"], item["reference"]
        context = "\n\n".join(d.page_content for d in retriever.invoke(q))
        #print("*****" * 10)
        #print(f"\n context (string): {context}")
        answer = chain.invoke(q)
        rows.append({
            "question": q,
            "answer": answer,
            "faithful": judge_faithfulness(answer, context).verdict,
            "relevant": judge_relevance(q, answer).verdict,
            "correct": judge_correctness(q, ref, answer).verdict,
        })
    return pd.DataFrame(rows)

In [16]:
results_k2 = evaluate(k=2)

In [17]:
results_k2

,question,answer,faithful,relevant,correct
0,Was ist Overfitting?,"Overfitting tritt auf, wenn ein Modell die Tra...",False,True,True
1,Was ist der Unterschied zwischen L1- und L2-Re...,Der Unterschied zwischen L1- und L2-Regularisi...,True,True,True
2,Warum ist Accuracy bei unausgewogenen Klassen ...,Bei unausgewogenen Klassen kann Accuracy irref...,True,True,True
3,Was ist Backpropagation?,"Backpropagation ist ein Algorithmus, der nach ...",True,True,True
4,Welches Problem haben einfache RNNs bei langen...,Einfache RNNs haben bei langen Sequenzen das P...,True,True,True
5,Wofür werden Convolutional Neural Networks (CN...,Convolutional Neural Networks (CNNs) werden fü...,True,True,True
6,Was ist Dropout?,Diese Information ist nicht in meinen Dokument...,False,False,False
7,Was bedeutet Data Drift und Concept Drift?,Data Drift bezeichnet eine Veränderung der sta...,True,True,True
8,Was ist ein Standardwert (Default-Argument) in...,Ein Standardwert (Default-Argument) in Python-...,True,True,True
9,Was ist Transfer Learning?,"Transfer Learning ist eine Standardmethode, di...",True,True,True


In [18]:
verdict_cols = ["faithful", "relevant", "correct"]

In [19]:
results_k2[verdict_cols].mean()

faithful    0.8
relevant    0.9
correct     0.9
dtype: float64


## Der Verbesserungszyklus

- **Eine Sache** ändern, neu bewerten, vergleichen
- Hier: mehr Kontext abrufen (`k=4` statt `k=2`)
- Mehr Kontext kann die Faithfulness verbessern — oder Rauschen hinzufügen

In [20]:
results_k4 = evaluate(k=4)
results_k4[verdict_cols].mean()

faithful    1.0
relevant    1.0
correct     1.0
dtype: float64

In [21]:
comparison = pd.DataFrame({
    "k=2": results_k2[verdict_cols].mean(),
    "k=4": results_k4[verdict_cols].mean(),
})

In [22]:
comparison

,k=2,k=4
faithful,0.8,1.0
relevant,0.9,1.0
correct,0.9,1.0



## Eine zweite Meinung: das Judge-Modell austauschen

- Der Judge muss nicht das RAG-Modell sein — der Austausch ist **eine Zeile**
- Ein anderes Modell umgeht die Selbstbevorzugung und liefert eine unabhängige Sicht

In [23]:
judge2_llm = create_judge_llm("openai/gpt-4.1-nano")
judge_faithfulness2, judge_relevance2, judge_correctness2 = make_judges(judge2_llm)

In [24]:
results_k2["relevant_judge2"] = [
    judge_relevance2(q, a).verdict
    for q, a in zip(results_k2["question"], results_k2["answer"])
]

In [25]:
results_k2

,question,answer,faithful,relevant,correct,relevant_judge2
0,Was ist Overfitting?,"Overfitting tritt auf, wenn ein Modell die Tra...",False,True,True,True
1,Was ist der Unterschied zwischen L1- und L2-Re...,Der Unterschied zwischen L1- und L2-Regularisi...,True,True,True,True
2,Warum ist Accuracy bei unausgewogenen Klassen ...,Bei unausgewogenen Klassen kann Accuracy irref...,True,True,True,True
3,Was ist Backpropagation?,"Backpropagation ist ein Algorithmus, der nach ...",True,True,True,True
4,Welches Problem haben einfache RNNs bei langen...,Einfache RNNs haben bei langen Sequenzen das P...,True,True,True,True
5,Wofür werden Convolutional Neural Networks (CN...,Convolutional Neural Networks (CNNs) werden fü...,True,True,True,True
6,Was ist Dropout?,Diese Information ist nicht in meinen Dokument...,False,False,False,False
7,Was bedeutet Data Drift und Concept Drift?,Data Drift bezeichnet eine Veränderung der sta...,True,True,True,True
8,Was ist ein Standardwert (Default-Argument) in...,Ein Standardwert (Default-Argument) in Python-...,True,True,True,True
9,Was ist Transfer Learning?,"Transfer Learning ist eine Standardmethode, di...",True,True,True,True


In [26]:
def evaluate(k):
    retriever, chain = build_rag_chain(vectorstore, llm, k, language=LANGUAGE)
    rows = []
    for item in test_set:
        q, ref = item["question"], item["reference"]
        context = "\n\n".join(d.page_content for d in retriever.invoke(q))
        answer = chain.invoke(q)
        rows.append({
            "question": q,
            "answer": answer,
            "faithful": judge_faithfulness2(answer, context).verdict,
            "relevant": judge_relevance2(q, answer).verdict,
            "correct": judge_correctness2(q, ref, answer).verdict,
        })
    return pd.DataFrame(rows)

In [27]:
results_k2_2 = evaluate(k=2)

In [28]:
comparison = pd.DataFrame({
    "k=2": results_k2[verdict_cols].mean(),
    "k=4": results_k4[verdict_cols].mean(),
    "k=2_2": results_k2_2[verdict_cols].mean()
})
comparison

,k=2,k=4,k=2_2
faithful,0.8,1.0,0.9
relevant,0.9,1.0,0.9
correct,0.9,1.0,0.9


### Schritt 3: evaluation using DeepEval

In [29]:
# Muss vor dem Import von deepeval gesetzt werden; langsame OpenRouter-Antworten brauchen mehr als 88 s
os.environ.setdefault("DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE", "300")

from deepeval.models import OpenRouterModel


In [30]:
class NoLogprobsOpenRouterModel(OpenRouterModel):
    # GEval fragt sonst Logprobs an, die das Modell über OpenRouter nicht liefert -> Hänger/Timeout
    def generate_raw_response(self, *args, **kwargs):
        raise AttributeError("log_probs unsupported.")


judge_model = NoLogprobsOpenRouterModel(
    model="ibm-granite/granite-4.2-8b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
    # granite-4.2 reasons by default; "minimal" keeps the judge's JSON valid
    # without spending the whole token budget on a hidden trace
    generation_kwargs={"extra_body": {"reasoning": {"effort": "minimal"}}},
)


In [31]:
faithfulness = FaithfulnessMetric(model=judge_model, async_mode=False)
relevancy = AnswerRelevancyMetric(model=judge_model, async_mode=False)
contextual_relevancy = ContextualRelevancyMetric(model=judge_model, async_mode=False)
contextual_recall =  ContextualRecallMetric(model=judge_model, async_mode=False)

In [32]:
correctness = GEval(
    name="Correctness",
    criteria="The actual output is correct if it is consistent with the "
    "expected output. Minor wording differences are acceptable. If the "
    "expected output says the information is not available, the actual "
    "output should indicate the same.",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=judge_model,
    async_mode=False,
)

In [33]:
retriever, chain = build_rag_chain(vectorstore, llm, k=2, language=LANGUAGE)

In [34]:
eval_data = []
for item in test_set:
    q = item["question"]
    contexts = [d.page_content for d in retriever.invoke(q)]
    answer = chain.invoke(q)
    eval_data.append(
        {"question": q, "contexts": contexts, "answer": answer,
         "reference": item["reference"]}
    )

In [35]:
deepeval_rows = []
for d in eval_data:
    case = LLMTestCase(
        input=d["question"],
        actual_output=d["answer"],
        retrieval_context=d["contexts"],
        expected_output=d["reference"],
    )
    # _show_indicator=False unterdrückt den rich-Fortschrittsbalken (Widget-Meldungen im Notebook)
    faithfulness.measure(case, _show_indicator=False)
    relevancy.measure(case, _show_indicator=False)
    correctness.measure(case, _show_indicator=False)
    contextual_relevancy.measure(case, _show_indicator=False)
    contextual_recall.measure(case, _show_indicator=False)
    deepeval_rows.append({
        "question": d["question"],
        "faithfulness": faithfulness.score,
        "answer_relevancy": relevancy.score,
        "contextual_relevancy": contextual_relevancy.score,
        "contextual_recall": contextual_recall.score,
        "correctness": correctness.score,
    })



In [36]:
pd.DataFrame(deepeval_rows)

,question,faithfulness,answer_relevancy,contextual_relevancy,contextual_recall,correctness
0,Was ist Overfitting?,1.0,1.0,0.700000,1.0,0.0
1,Was ist der Unterschied zwischen L1- und L2-Re...,1.0,1.0,0.333333,1.0,0.6
2,Warum ist Accuracy bei unausgewogenen Klassen ...,1.0,1.0,0.125000,1.0,0.6
3,Was ist Backpropagation?,1.0,1.0,0.272727,1.0,0.4
4,Welches Problem haben einfache RNNs bei langen...,1.0,1.0,0.700000,1.0,0.9
5,Wofür werden Convolutional Neural Networks (CN...,1.0,1.0,1.000000,1.0,0.0
6,Was ist Dropout?,1.0,0.0,0.000000,0.0,0.0
7,Was bedeutet Data Drift und Concept Drift?,1.0,1.0,0.600000,1.0,0.7
8,Was ist ein Standardwert (Default-Argument) in...,1.0,1.0,0.333333,1.0,0.7
9,Was ist Transfer Learning?,1.0,1.0,0.666667,1.0,0.4


### RAGAS Metriken

In [37]:
""" RAGAS evaluation metrics """
import sys
import types

# ragas 0.4.3 importiert ChatVertexAI aus langchain_community, das es dort nicht mehr gibt -> Stub
_stub = types.ModuleType("langchain_community.chat_models.vertexai")
_stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules.setdefault("langchain_community.chat_models.vertexai", _stub)

# evaluate wird umbenannt, weil das Notebook bereits eine eigene evaluate(k) definiert
from ragas import EvaluationDataset, evaluate as ragas_evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ResponseRelevancy,
)
from ragas.run_config import RunConfig

C:\Users\goran\AppData\Local\Temp\ipykernel_22120\2788882737.py:14: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\goran\AppData\Local\Temp\ipykernel_22120\2788882737.py:14: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\goran\AppData\Local\Temp\ipykernel_22120\2788882737.py:14: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\gora

In [38]:
# Judge-LLM aus create_rag_system, Embeddings aus dem QdrantVectorStore
ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(vectorstore.embeddings)

# eval_data (Frage, Kontexte, Antwort, Referenz) stammt aus dem DeepEval-Abschnitt (k=2)
ragas_samples = [
    {
        "user_input": d["question"],
        "retrieved_contexts": d["contexts"],
        "response": d["answer"],
        "reference": d["reference"],
    }
    for d in eval_data
]

ragas_metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
    LLMContextPrecisionWithReference(llm=ragas_llm),
    LLMContextRecall(llm=ragas_llm),
]

ragas_result = ragas_evaluate(
    dataset=EvaluationDataset.from_list(ragas_samples),
    metrics=ragas_metrics,
    run_config=RunConfig(timeout=180, max_workers=4),
)

C:\Users\goran\AppData\Local\Temp\ipykernel_22120\663654817.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(judge_llm)
C:\Users\goran\AppData\Local\Temp\ipykernel_22120\663654817.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(vectorstore.embeddings)


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[13]: TypeError(Chat.send_async() got an unexpected keyword argument 'n')
Exception raised in Job[17]: TypeError(Chat.send_async() got an unexpected keyword argument 'n')
Exception raised in Job[9]: TypeError(Chat.send_async() got an unexpected keyword argument 'n')
Exception raised in Job[21]: TypeError(Chat.send_async() got an unexpected keyword argument 'n')
Exception raised in Job[25]: TypeError(Chat.send_async() got an unexpected keyword argument 'n')
Exception raised in Job[37]: TypeError(Chat.send_async() got an unexpected keyword argument 'n')
Exception raised in Job[33]: TypeError(Chat.send_async() got an unexpected keyword argument 'n')
Exception raised in Job[29]: TimeoutError()


In [39]:
ragas_df = ragas_result.to_pandas()
metric_cols = [c for c in ragas_df.columns if c in
               ("faithfulness", "answer_relevancy", "llm_context_precision_with_reference", "context_recall")]
display(ragas_df[["user_input", *metric_cols]])
print("Mittelwerte:")
print(ragas_df[metric_cols].mean().round(3))

,user_input,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,Was ist Overfitting?,0.75,0.739656,1.0,1.0
1,Was ist der Unterschied zwischen L1- und L2-Re...,0.80,0.695950,1.0,1.0
2,Warum ist Accuracy bei unausgewogenen Klassen ...,1.00,NaN,1.0,1.0
3,Was ist Backpropagation?,1.00,NaN,1.0,1.0
4,Welches Problem haben einfache RNNs bei langen...,1.00,NaN,1.0,1.0
5,Wofür werden Convolutional Neural Networks (CN...,1.00,NaN,1.0,1.0
6,Was ist Dropout?,0.00,NaN,0.0,0.0
7,Was bedeutet Data Drift und Concept Drift?,1.00,NaN,1.0,1.0
8,Was ist ein Standardwert (Default-Argument) in...,0.70,NaN,1.0,1.0
9,Was ist Transfer Learning?,0.40,NaN,1.0,1.0


Mittelwerte:
faithfulness                            0.765
answer_relevancy                        0.718
llm_context_precision_with_reference    0.900
context_recall                          0.900
dtype: float64
